In [9]:
import config
import json
import pandas as pd
import matplotlib.pyplot as plt
import itertools
from collections import defaultdict

In [10]:
%config InlineBackend.figure_format = 'retina'

In [11]:
with open(f"{config.DIR_REPBASE_PROCESSED}/metadata_02_ltr_correction.json", "r", encoding="utf-8") as f:
    data = json.load(f)

# with open(f"{config.DIR_REPBASE_PROCESSED}/translation_dict.json", "r", encoding="utf-8") as f:
#     translation_dict = json.load(f)

In [12]:
TARGET_FRAC = 0.2

In [13]:
import json
from collections import defaultdict

# === organism -> set(classes), organism -> number of records ===
org_to_classes = defaultdict(set)
org_to_count = defaultdict(int)

for te_name, info in data.items():
    org = info["organism"]
    cls = info["class"]
    org_to_classes[org].add(cls)
    org_to_count[org] += 1

all_classes = set()
for classes in org_to_classes.values():
    all_classes |= classes

total_records = sum(org_to_count.values())
target_records = total_records * TARGET_FRAC

remaining = set(org_to_classes.keys())
selected = []
covered = set()
current_count = 0

# Сначала покрываем все классы
while covered != all_classes and remaining:
    best_org = None
    best_key = None

    for org in remaining:
        new_classes = org_to_classes[org] - covered
        gain = len(new_classes)
        count = org_to_count[org]

        if gain == 0:
            continue

        # лучше брать организм, который:
        # 1) добавляет больше новых классов
        # 2) при равенстве имеет меньше записей
        # 3) при равенстве дает лучший gain/count
        key = (gain, -count, gain / count)

        if best_key is None or key > best_key:
            best_key = key
            best_org = org

    if best_org is None:
        break

    selected.append(best_org)
    remaining.remove(best_org)
    covered |= org_to_classes[best_org]
    current_count += org_to_count[best_org]

# Если покрытие уже есть, можно чуть подогнать размер к TARGET_FRAC
# добавляем организмы, которые приближают размер к target_records
improved = True
while improved and remaining:
    improved = False
    current_diff = abs(current_count - target_records)

    best_org = None
    best_diff = current_diff

    for org in remaining:
        new_count = current_count + org_to_count[org]
        new_diff = abs(new_count - target_records)

        if new_diff < best_diff:
            best_diff = new_diff
            best_org = org

    if best_org is not None:
        selected.append(best_org)
        remaining.remove(best_org)
        current_count += org_to_count[best_org]
        improved = True

val_organisms = set(selected)

# === разбиение ID ===
train_ids = []
test_ids = []

for te_name, info in data.items():
    if info["organism"] in val_organisms:
        test_ids.append(te_name)
    else:
        train_ids.append(te_name)

train_ids.sort()
test_ids.sort()

# === сохранение ===
with open(f"{config.DIR_REPBASE_PROCESSED}/id_train.txt", "w", encoding="utf-8") as f:
    for seq_id in train_ids:
        f.write(seq_id + "\n")

with open(f"{config.DIR_REPBASE_PROCESSED}/id_test.txt", "w", encoding="utf-8") as f:
    for seq_id in test_ids:
        f.write(seq_id + "\n")

# === отчет ===
covered_final = set()
for org in val_organisms:
    covered_final |= org_to_classes[org]

print(f"Выбрано организмов в test: {len(val_organisms)}")
print(f"Всего записей: {total_records}")
print(f"Test записей: {len(test_ids)}")
print(f"Train записей: {len(train_ids)}")
print(f"Доля test: {len(test_ids) / total_records:.4f}")
print(f"Цель: {TARGET_FRAC:.4f}")
print(f"Покрытие классов: {len(covered_final)} / {len(all_classes)}")

print("\nОрганизмы в test:")
for org in sorted(val_organisms):
    print(f"{org}\t{org_to_count[org]}")

Выбрано организмов в test: 12
Всего записей: 120813
Test записей: 24164
Train записей: 96649
Доля test: 0.2000
Цель: 0.2000
Покрытие классов: 29 / 29

Организмы в test:
Aedes aegypti	4971
Chondrus crispus	1385
Crassostrea gigas	2041
Danio rerio	2445
Nematostella vectensis	797
Oryza sativa	3613
Petromyzon marinus	249
Sphenodon punctatus	2013
Triticum aestivum	1827
Xenopus laevis	2308
Zea mays	2455
Zymoseptoria tritici	60


Создадим отдельный файл для train

In [15]:
with open(f"{config.DIR_REPBASE_PROCESSED}/id_train.txt", "r", encoding="utf-8") as f:
    ids_train = [line.strip() for line in f]

In [37]:
import json

with open(f"{config.DIR_REPBASE_PROCESSED}/hierarchy_sequences_02_ltr_correction.json", "r", encoding="utf-8") as f:
    hierarchy_sequences = json.load(f)

In [21]:
def filter_ids(l, ids):
    return [x for x in l if x in ids]

In [23]:
hierarchy_sequences_train = hierarchy_sequences.copy()
for name, d in hierarchy_sequences_train.items():
    if not d["sequences"]:
        continue
    d["sequences"] = filter_ids(d["sequences"], ids_train)
    for sub_name, sub_d in d["subs"].items():
        if not sub_d["sequences"]:
            continue
        sub_d["sequences"] = filter_ids(sub_d["sequences"], ids_train)

In [27]:
with open(f"{config.DIR_REPBASE_PROCESSED}/hierarchy_sequences_02_ltr_correction_train.json", "w", encoding="utf-8") as f:
    json.dump(hierarchy_sequences_train, f, ensure_ascii=False, indent=2)

Создадим отдельный файл для test

In [34]:
with open(f"{config.DIR_REPBASE_PROCESSED}/id_test.txt", "r", encoding="utf-8") as f:
    ids_test = [line.strip() for line in f]

In [35]:
ids_test

['AACOPIA1_I',
 'AACOPIA1_LTR',
 'AC9_ZM',
 'ACROBAT1',
 'ACROBAT2',
 'ALBERT_TA',
 'AMYLTP',
 'ANGEL',
 'ATLANTYS-I_OS',
 'ATLANTYS-LTR_OS',
 'ATLANTYS_OS',
 'Academ-10_CCri',
 'Academ-11_CCri',
 'Academ-12_CCri',
 'Academ-13_CCri',
 'Academ-14N1_CCri',
 'Academ-14_CCri',
 'Academ-15_CCri',
 'Academ-1_CCri',
 'Academ-1_CGi',
 'Academ-1_DR',
 'Academ-1_NV',
 'Academ-1_PM',
 'Academ-1_SpPu',
 'Academ-2_CCri',
 'Academ-2_CGi',
 'Academ-2_PM',
 'Academ-2_SpPu',
 'Academ-3_CCri',
 'Academ-3_CGi',
 'Academ-4B_CGi',
 'Academ-4_CCri',
 'Academ-4_CGi',
 'Academ-5_CCri',
 'Academ-5_CGi',
 'Academ-6_CCri',
 'Academ-7_CCri',
 'Academ-8_CCri',
 'Academ-9_CCri',
 'Academ-N1_DRe',
 'Academ-N1_NV',
 'Academ-N2_NV',
 'Academ-N3_NV',
 'Academ-N4_NV',
 'Academ-N5_NV',
 'AcademH-10_CGi',
 'AcademH-11N1_CGi',
 'AcademH-11_CGi',
 'AcademH-12_CGi',
 'AcademH-13N1_CGi',
 'AcademH-13_CGi',
 'AcademH-14_CGi',
 'AcademH-15_CGi',
 'AcademH-16_CGi',
 'AcademH-17_CGi',
 'AcademH-18N1_CGi',
 'AcademH-18_CGi',
 'Aca

In [26]:
bool(set(ids_train) & set(ids_test))

False

In [38]:
hierarchy_sequences_test = hierarchy_sequences.copy()
for name, d in hierarchy_sequences_test.items():
    if not d["sequences"]:
        continue
    d["sequences"] = filter_ids(d["sequences"], ids_test)
    for sub_name, sub_d in d["subs"].items():
        if not sub_d["sequences"]:
            continue
        sub_d["sequences"] = filter_ids(sub_d["sequences"], ids_test)

In [39]:
hierarchy_sequences_test

{'DNA transposon': {'sequences': ['DNA-X-3_NV',
   'DNA-5_NV',
   'DNA-9-22_NV',
   'DNA-TA-20A_NV',
   'DNA-7-11_NV',
   'DNA-TA-24_NV',
   'DNA-TA-9A_NV',
   'DNA-X-2_NV',
   'DNA-TA-13_NV',
   'Harbinger-N25_NV',
   'DNA-TTAA-5_NV',
   'DNA-TA-26A_NV',
   'DNA-9-27_NV',
   'DNA-9-29_NV',
   'DNA-11_NV',
   'DNA-9-6_NV',
   'DNA-7-4_NV',
   'DNA-7-23_NV',
   'DNA-3_NV',
   'DNA-9-30_NV',
   'DNA-TA-16_NV',
   'DNA-TA-22_NV',
   'DNA-6-8_NV',
   'DNA-4-1_NV',
   'DNA-7-17_NV',
   'DNA-9-24_NV',
   'DNA-9-20_NV',
   'DNA-2-6_NV',
   'DNA-7-14_NV',
   'DNA-TA-21-LTR_NV',
   'DNA-10_NV',
   'DNA-6-10_NV',
   'DNA-6-2A_NV',
   'DNA-7-18_NV',
   'DNA-7-21A_NV',
   'DNA-7-10_NV',
   'DNA-TA-8_NV',
   'DNA-9-14_NV',
   'DNA-5-20_NV',
   'piggyBac-N6_NV',
   'DNA-9-31_NV',
   'DNA-TA-15_NV',
   'DNA-X-10_NV',
   'DNA-3-10_NV',
   'DNA-5-19_NV',
   'DNA-TWA-1_NV',
   'DNA-7-15_NV',
   'DNA-TA-2_NV',
   'DNA-9-23A_NV',
   'DNA-TTAA-7_NV',
   'DNA-7-8_NV',
   'DNA-7-21_NV',
   'DNA-9-8_NV',
   '

In [40]:
with open(f"{config.DIR_REPBASE_PROCESSED}/hierarchy_sequences_02_ltr_correction_test.json", "w", encoding="utf-8") as f:
    json.dump(hierarchy_sequences_test, f, ensure_ascii=False, indent=2)

Проверяем хоть какое-то пересечение

In [31]:
from typing import Any, Iterable, Set


def extract_sequences(node: Any) -> Set[str]:
    """
    Рекурсивно собирает все элементы из поля `sequences`
    по всему иерархическому словарю.

    Поддерживает варианты:
    - sequences = list[str]
    - sequences = dict[..., str] / dict[..., ...]
    - subs = dict[str, node]
    """
    result = set()

    def walk(obj: Any):
        if not isinstance(obj, dict):
            return

        seqs = obj.get("sequences", [])
        if isinstance(seqs, dict):
            for k, v in seqs.items():
                if isinstance(k, str):
                    result.add(k)
                if isinstance(v, str):
                    result.add(v)
                elif isinstance(v, (list, tuple, set)):
                    result.update(map(str, v))
        elif isinstance(seqs, (list, tuple, set)):
            result.update(map(str, seqs))
        elif isinstance(seqs, str):
            result.add(seqs)

        subs = obj.get("subs", {})
        if isinstance(subs, dict):
            for child in subs.values():
                walk(child)

    walk(node)
    return result


def has_any_intersection(tree1: dict, tree2: dict) -> bool:
    seqs1 = extract_sequences(tree1)
    seqs2 = extract_sequences(tree2)
    return not seqs1.isdisjoint(seqs2)


def get_intersection(tree1: dict, tree2: dict) -> Set[str]:
    seqs1 = extract_sequences(tree1)
    seqs2 = extract_sequences(tree2)
    return seqs1 & seqs2

In [41]:
common = get_intersection(hierarchy_sequences_train, hierarchy_sequences_test)
print("Количество общих элементов:", len(common))

if common:
    print("Примеры:", list(common)[:10])

Количество общих элементов: 0
